# Solar Rooftop Potential — LoD2 Building Approach

Compute solar thermal and PV potential using **LoD2 (Level of Detail 2) building models** with actual roof geometry (slope, orientation, area). This approach requires 3D building data but provides more accurate results. Includes an **oemof-based sensitivity analysis** of system costs and CO\N{GREEK SMALL LETTER ALPHA} emissions.

---

## 1. Imports

In [ ]:
import atlite
import json
import numpy as np
import geopandas as gpd
import rasterio as rio
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import hvplot.pandas

from src.data_loading import load_shape, load_demand, load_cop, load_cutout
from src.lod2_processing import (
    load_lod2_buildings,
    clip_buildings_to_region,
    rasterize_buildings,
    average_roof_params,
)

hvplot.extension("bokeh")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})


## 2. File Paths

All input data lives in `data/`.

In [ ]:
shape_path = "data/leeste3035.gpkg"
demand_path = "data/Last_240222.csv"
cop_path = "data/COP_WP.xlsx"
era5_cutout_path = "data/era5-2014-leeste.nc"

lod2_files = [
    "data/lod2_1.gpkg",
    "data/lod2_2.gpkg",
    "data/lod2_3.gpkg",
]


## 3. Load Region Boundary

In [ ]:
shape = load_shape(shape_path)
shape = shape.set_index("Name")

minx, miny, maxx, maxy = shape.total_bounds
buffer = 0.005
resolution = 5  # metres per pixel (for LoD2 rasterization)
x_slice = slice(minx - buffer, maxx + buffer)
y_slice = slice(miny - buffer, maxy + buffer)


## 4. Load Demand & COP Data

In [ ]:
load_dhw, load_sph = load_demand(demand_path, year=2014)
cop = load_cop(cop_path, year=2014)


## 5. ERA5 Weather Cutout

In [ ]:
cutout = load_cutout(era5_cutout_path, x_slice, y_slice, year=2014)
# cutout.prepare()  # uncomment to download ERA5 data


## 6. Load and Process LoD2 Building Data

Load three LoD2 GeoPackage tiles (German data format), concatenate, reproject, and clip to the region boundary. The roof attributes (`Dachneigung`, `Dachorientierung`, `Dachflaeche`) are JSON-encoded in the source data and are parsed to numeric values.

In [ ]:
buildings = load_lod2_buildings(lod2_files, crs="25832")
print(f"Loaded {len(buildings)} building features.")
print(f"Original CRS: {buildings.crs}")

buildings = buildings.to_crs(3035)
buildings = clip_buildings_to_region(buildings, shape)
print(f"Filtered to {len(buildings)} features within region.")

total_roof_area = buildings["Dachflaeche"].sum()
print(f"Total roof area: {total_roof_area:.1f} m\u00b2")


## 7. Rasterize Building Footprints

Convert the vector building footprints to a raster for use with atlite's `ExclusionContainer`. The raster is saved to a temporary file.

In [ ]:
temp_raster = "temp_buildings.tif"
raster, transform = rasterize_buildings(
    buildings, (minx, miny, maxx, maxy), resolution, temp_raster
)

buildings_excluder = atlite.ExclusionContainer(crs=3035, res=resolution)
buildings_excluder.add_raster(temp_raster, codes=1, invert=True)

band1, transform1 = atlite.gis.shape_availability(shape.geometry, buildings_excluder)

cell_area = abs(transform1.a * transform1.e)
excluded_cells = np.count_nonzero(band1 == 1)
total_excluded = excluded_cells * cell_area
print(f"Available roof area in region: {total_excluded:.0f} m\u00b2")


## 8. Average Roof Slope & Azimuth

Compute the mean roof slope and azimuth from all buildings in the region. These will be used as the PV panel orientation for a more realistic simulation.

In [ ]:
avg_slope, avg_azimuth = average_roof_params(buildings)
print(f"Average slope: {avg_slope:.1f}\u00b0, Average azimuth: {avg_azimuth:.1f}\u00b0")


## 9. Availability Matrix

Uses `cap_per_sqkm = 19` MW/km\u00b2 (a higher modern value) and the LoD2-derived building raster.

In [ ]:
A = cutout.availabilitymatrix(shape, buildings_excluder)

cap_per_sqkm = 19  # MW/km\u00b2 (modern high-density assumption)
area = cutout.grid.set_index(["y", "x"]).to_crs(3035).area / 1e6
area = xr.DataArray(area, dims=("spatial",))
capacity_matrix = A.stack(spatial=["y", "x"]) * area * cap_per_sqkm

band, transform = atlite.gis.shape_availability(shape.geometry, buildings_excluder)

fig, ax = plt.subplots(figsize=(8, 8))
shape.plot(ax=ax, color="none")
ax.set_xlabel("Eastings (m)")
ax.set_ylabel("Northings (m)")
rio.plot.show(band, transform=transform, cmap="Greens", ax=ax)
plt.title("Available Roof Area (LoD2-derived)")
plt.show()


## 10. Solar Thermal Potential

In [ ]:
csp_potential_year = cutout.csp(
    installation="lossless_installation",
    technology="parabolic trough",
    index=shape.index,
    matrix=capacity_matrix,
)

csp_potential_year = xr.concat(csp_potential_year, dim="time")
st_potential = csp_potential_year.sum(dim=["Name"]) * (25 / 60)
st_potential = st_potential.to_pandas()

coverage_st = st_potential.sum() / load_dhw.sum() * 100
print(f"Solar thermal covers {coverage_st:.1f}% of DHW demand")


## 11. PV Potential

Two PV simulations are run:
1. **LoD2 orientation** — panels follow the actual average roof slope/azimuth
2. **Latitude-optimal** — panels at the ideal tilt (for comparison)

In [ ]:
pv = cutout.pv(
    panel=atlite.solarpanels.CdTe,
    orientation=dict(slope=avg_slope, azimuth=avg_azimuth),
    matrix=capacity_matrix,
    index=shape.index,
)
pv = pv.to_pandas()
pv = pv["LeesteFull"]

pv_max = cutout.pv(
    panel=atlite.solarpanels.CdTe,
    orientation="latitude_optimal",
    matrix=capacity_matrix,
    index=shape.index,
)
pv_max = pv_max.to_pandas()
pv_max = pv_max["LeesteFull"]

ratio = pv.sum() / pv_max.sum() * 100
print(f"LoD2 orientation yields {ratio:.1f}% of latitude-optimal PV")
coverage_pv = pv.sum() / load_sph.sum() * 100
print(f"PV covers {coverage_pv:.1f}% of space heating demand (via HP)")


## 12. PV-Powered Heat Pump

In [ ]:
hp = pv * cop


## 13. Time-Series Plots

### 13.1 Solar Thermal Potential

In [ ]:
st_potential.plot.line(
    linewidth=0.5, figsize=(14, 4),
    xlabel="Time", ylabel="Solar Thermal Potential (MW)",
    title="Solar Thermal Potential"
)
plt.show()


### 13.2 Domestic Hot Water Demand

In [ ]:
load_dhw.plot.line(
    linewidth=0.5, figsize=(14, 4),
    xlabel="Time", ylabel="DHW Demand (MW)",
    title="Domestic Hot Water Load"
)
plt.show()


### 13.3 Solar Thermal — DHW Mismatch

In [ ]:
diff1 = st_potential - load_dhw
diff1.plot.area(figsize=(14, 4), xlabel="Time",
               ylabel="Mismatch (MW)",
               title="Solar Thermal to DHW: Production minus Consumption")
plt.show()

diff1.resample("D").mean().plot.area(
    figsize=(14, 4), xlabel="Time",
    ylabel="Mismatch (MW)",
    title="Daily Average Mismatch"
)
plt.show()


### 13.4 PV Potential

In [ ]:
pv.plot.line(linewidth=0.5, figsize=(14, 4),
             xlabel="Time", ylabel="PV Potential (MW)",
             title="PV Potential (LoD2 Orientation)")
plt.show()


### 13.5 Heat Pump Thermal Output

In [ ]:
hp.plot.line(linewidth=0.5, figsize=(14, 4),
             xlabel="Time", ylabel="Thermal Power (MW)",
             title="Air-Source Heat Pump Thermal Output")
plt.show()


### 13.6 Space Heating Demand

In [ ]:
load_sph.plot.line(linewidth=0.5, figsize=(14, 4),
                    xlabel="Time", ylabel="Space Heating Load (MW)",
                    title="Space Heating Load")
plt.show()


### 13.7 Heat Pump — Space Heating Mismatch

In [ ]:
diff2 = hp - load_sph
diff2.plot.area(figsize=(14, 4), xlabel="Time",
               ylabel="Mismatch (MW)",
               title="HP Production minus Space Heating Demand")
plt.show()

diff2.resample("D").mean().plot.area(
    figsize=(14, 4), xlabel="Time",
    ylabel="Mismatch (MW)",
    title="Daily Average Mismatch"
)
plt.show()


## 14. Sensitivity Analysis ($\alpha$ Variation)

In [ ]:
alpha = [0, 0.2, 0.4, 0.6, 0.8, 1]
st_a, hp_a = {}, {}
demand_cover_dw_a, demand_cover_sh_a = {}, {}

for a in alpha:
    pv_temp = pv * a
    st_temp = st_potential * (1 - a)
    hp_temp = pv_temp * cop

    diff1_t = st_temp - load_dhw
    diff2_t = hp_temp - load_sph

    st_a[a] = diff1_t[diff1_t > 0].sum()
    hp_a[a] = diff2_t[diff2_t > 0].sum()

    demand_cover_dw_a[a] = st_temp.sum() / load_dhw.sum() * 100
    demand_cover_sh_a[a] = hp_temp.sum() / load_sph.sum() * 100

print("Overproduction DHW (MWh):", st_a)
print("Overproduction SH  (MWh):", hp_a)
print("DHW coverage       (%):", demand_cover_dw_a)
print("SH  coverage       (%):", demand_cover_sh_a)


### 14.1 Load Coverage Plot

In [ ]:
keys = list(demand_cover_dw_a.keys())
v1 = list(demand_cover_dw_a.values())
v2 = list(demand_cover_sh_a.values())

fig, ax = plt.subplots(figsize=(7, 5))
ax.fill_between(keys, v1, alpha=0.5, label="Domestic Hot Water")
ax.fill_between(keys, v2, alpha=0.5, label="Space Heating")
ax.set_xlabel("$\\alpha$")
ax.set_ylabel("Load Coverage [%]")
ax.set_title("Sensitivity Analysis — Load Coverage")
ax.legend(loc="upper left")
ax.grid(True)
plt.show()


### 14.2 Overproduction Plot

In [ ]:
v3 = np.array(list(st_a.values())) / load_dhw.sum() * 100
v4 = np.array(list(hp_a.values())) / load_sph.sum() * 100 / cop.mean()

fig, ax = plt.subplots(figsize=(7, 5))
ax.fill_between(keys, v3, alpha=0.5, label="Domestic Hot Water")
ax.fill_between(keys, v4, alpha=0.5, label="PV")
ax.set_xlabel("$\\alpha$")
ax.set_ylabel("Overproduction / Yearly Demand [%]")
ax.set_title("Sensitivity Analysis — Overproduction")
ax.legend(loc="upper left")
ax.grid(True)
plt.show()


## 15. oemof Sensitivity Analysis

Pre-computed results from an oemof (Open Energy Modelling Framework) optimization showing total system costs and CO\N{GREEK SMALL LETTER ALPHA} emissions for each $\alpha$ scenario. These values were generated by a separate oemof energy system model.

In [ ]:
keys = list(demand_cover_dw_a.keys())
oemof_cost = np.array([346307, 337263, 329143, 321346, 313729, 306247]) / 1000
oemof_co2 = np.array([404172, 388089, 371529, 355043, 338592, 321834]) / 1000

fig, ax = plt.subplots(figsize=(7, 5))
ax.fill_between(keys, oemof_cost, alpha=0.7, label="System Costs")
ax.fill_between(keys, oemof_co2, alpha=0.5, label="CO\N{GREEK SMALL LETTER EPSILON} Emissions")
ax.set_xlabel("$\\alpha$")
ax.set_ylabel("Costs [k\u20ac], CO\N{GREEK SMALL LETTER EPSILON} Emissions [t]")
ax.set_title("Sensitivity Analysis with oemof")
ax.legend(loc="upper right")
ax.grid(True)
plt.show()
# fig.savefig("oemof_sensitivity.pdf", dpi=300)
